# Phase 3, Item 3 — Static Context Bundle

Proving each of the seven components before moving the logic into
`app/gateway/context.py` / `app/gateway/model_schema.py`. Starting with the
orientation bundle.

## Orientation bundle

Three pieces, already manually extracted into `context/orientation/`:
Executive Summary, Project Navigator, and a hand-drawn ASCII architecture
diagram (the styled HTML stays in the README, for humans —
`.claude/rules/gateway.md`). This step is just proving the assembly: read
the three files, join them into one block, sanity-check the result reads as
one coherent document and get a rough size estimate.

In [87]:
import pathlib

ORIENTATION_DIR = pathlib.Path("../context/orientation")
ORIENTATION_FILES = ["executive_summary.txt", "project_navigator.txt", "system_architecture.txt"]

ORIENTATION_PREAMBLE = """This section is background on the Instacart reorder-recommendation ML project
this dashboard evaluates, extracted from that project's own documentation --
not live data. Numbers in it (Recall@5, profit-lift figures, etc.) are
historical, point-in-time report figures; never cite them as an answer -- a
user-facing number always needs its own live run_bigquery_sql/run_dax_query
call this turn.

The Project Navigator is that documentation's table of contents. Each item
names a topic search_docs can retrieve in more depth -- treat it as an index
of what's available, not a substitute for looking something up. System
Architecture is the one exception: it's included here in full below, not
indexed separately, so search_docs won't return anything more on it.""".strip()


def build_orientation_bundle() -> str:
    sections = [(ORIENTATION_DIR / name).read_text(encoding="utf-8").strip() for name in ORIENTATION_FILES]
    return ORIENTATION_PREAMBLE + "\n\n" + "\n\n---\n\n".join(sections)


ORIENTATION_BUNDLE = build_orientation_bundle()
print(ORIENTATION_BUNDLE)

This section is background on the Instacart reorder-recommendation ML project
this dashboard evaluates, extracted from that project's own documentation --
not live data. Numbers in it (Recall@5, profit-lift figures, etc.) are
historical, point-in-time report figures; never cite them as an answer -- a
user-facing number always needs its own live run_bigquery_sql/run_dax_query
call this turn.

The Project Navigator is that documentation's table of contents. Each item
names a topic search_docs can retrieve in more depth -- treat it as an index
of what's available, not a substitute for looking something up. System
Architecture is the one exception: it's included here in full below, not
indexed separately, so search_docs won't return anything more on it.

EXECUTIVE SUMMARY

This project leverages the Kaggle Instacart Market Basket Analysis dataset to
build a machine learning-powered recommendation engine designed to populate a
personalized, 5-slot reorder widget during active shopping sessi

In [88]:
# Rough size check -- real token counts come once the whole seven-component
# block exists (gateway.md's minimum-cacheable-prefix note), but useful to
# see this piece's own footprint now.
chars = len(ORIENTATION_BUNDLE)
print(f"{chars} chars, ~{chars // 4} tokens (rough estimate, 4 chars/token)")

10495 chars, ~2623 tokens (rough estimate, 4 chars/token)


## TABLE_REGISTRY + MEASURE_REGISTRY + RELATIONSHIPS + PARAMETERS

All four render directly from `context/schema/model_schema.json` — no new
extraction, just format decisions. `MEASURE_REGISTRY` must carry names and
descriptions only, never the `dax` field (`.claude/rules/gateway.md`,
`.claude/rules/mcp-tools.md`) — that's fetched per-measure by
`get_measure_dax` instead.

In [89]:
import json

SCHEMA_PATH = pathlib.Path("../context/schema/model_schema.json")
schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))

print(f"{len(schema['tables'])} tables, {len(schema['measures'])} measures, "
      f"{len(schema['relationships'])} relationships, {len(schema['parameters'])} parameters")

24 tables, 60 measures, 8 relationships, 7 parameters


In [90]:
def _render_tables(tables: list[dict]) -> str:
    header = "TABLE REGISTRY -- every table in the semantic model, with its columns, types, and descriptions."
    blocks = []
    for t in tables:
        lines = [f"### {t['name']}"]
        if t.get("description"):
            lines.append(t["description"])
        for c in t["columns"]:
            desc = f": {c['description']}" if c.get("description") else ""
            lines.append(f"- {c['name']} ({c['type']}){desc}")
        blocks.append("\n".join(lines))
    return header + "\n\n" + "\n\n".join(blocks)


TABLE_REGISTRY = _render_tables(schema["tables"])
print(TABLE_REGISTRY[:1500])
print("...")
print(f"\n{len(TABLE_REGISTRY)} chars total")

TABLE REGISTRY -- every table in the semantic model, with its columns, types, and descriptions.

### beeswarm_plot_data
One row per prediction, per input feature — the underlying points for the SHAP beeswarm plot.
- unique_plot_id (Integer): unique plot id for beeswarm chart
- user_id (Integer): Instacart user ID for this prediction row
- anchor_order_number (Integer): Order number used as the prediction's anchor point for this user
- label_reordered (Integer): Actual outcome: 1 if the product was reordered, 0 if not
- prediction (Number): Model-predicted probability of reorder for this product
- feature_name (Text): Name of the input feature being visualized for this point
- feature_value (Number): Raw value of the feature for this row
- shap_value (Number): SHAP value: this feature's contribution to the prediction
- model_name (Text): Name of the model that produced this prediction
- model_id (Text): Unique identifier for model, 1:1 with model name
- split (Text): Dataset split this 

In [91]:
def _render_measures(measures: list[dict]) -> str:
    header = ("MEASURE REGISTRY -- every measure's name and description. "
               "Names and descriptions only -- fetch a measure's DAX body with "
               "get_measure_dax when you actually need to see its formula.")
    by_table: dict[str, list[dict]] = {}
    for m in measures:
        by_table.setdefault(m["table"], []).append(m)

    blocks = []
    for table, ms in by_table.items():
        lines = [f"### {table}"]
        for m in ms:
            desc = f": {m['description']}" if m.get("description") else ""
            lines.append(f"- {m['name']}{desc}")
        blocks.append("\n".join(lines))
    return header + "\n\n" + "\n\n".join(blocks)


MEASURE_REGISTRY = _render_measures(schema["measures"])
print(MEASURE_REGISTRY[:1200])
print("...")
print(f"\n{len(MEASURE_REGISTRY)} chars total")

MEASURE REGISTRY -- every measure's name and description. Names and descriptions only -- fetch a measure's DAX body with get_measure_dax when you actually need to see its formula.

### _Model_Evaluation_Measures
- Recall: Model Evaluation metric: Recall at 5
- Precision: Model evaluation metric: Precision at 5
- NDCG: Model evaluation metric: NDCG at 5
- F1: Model evaluation metric: F1 at 5
- Dynamic_Metric_Description: Plain-language definition of whichever evaluation metric is currently selected in Evaluation Metric Parameter
- Champion Recall: Recall at 5 for the champion model
- Champion Precision: Precision at 5 for the champion model
- Champion NDCG: NDCG at 5 for the champion model
- Champion F1: F1 at 5 for the champion model
- Feature_Y_Axis_Position: Vertical position for a beeswarm plot point, offset slightly within its feature's row to avoid perfect overlap
- SHAP_Color_Scale: This point's feature value, normalized 0–1 within that feature's own range, used to color the bees

In [92]:
def _render_relationships(relationships: list[dict]) -> str:
    header = "RELATIONSHIPS -- join paths between tables in the semantic model."
    lines = []
    for r in relationships:
        active = "active" if r["is_active"] else "inactive"
        lines.append(
            f"- {r['from_table']}[{r['from_column']}] -> {r['to_table']}[{r['to_column']}] "
            f"({r['from_cardinality']}:{r['to_cardinality']}, {r['cross_filtering_behavior']}, {active})"
        )
    return header + "\n" + "\n".join(lines)


RELATIONSHIPS = _render_relationships(schema["relationships"])
print(RELATIONSHIPS)

RELATIONSHIPS -- join paths between tables in the semantic model.
- beeswarm_plot_data[model_type] -> model_types_dimension[model_type] (Many:One, OneDirection, active)
- beeswarm_plot_data[split] -> dataset_split_dimension[dataset_split] (Many:One, OneDirection, active)
- beeswarm_plot_data[model_name] -> base_models_dimension[model_name] (Many:One, OneDirection, active)
- evaluation_metrics[Model] -> models_dimension[Model] (Many:One, OneDirection, active)
- evaluation_metrics[split] -> dataset_split_dimension[dataset_split] (Many:One, OneDirection, active)
- power_bi_shap_barplots[model_type] -> model_types_dimension[model_type] (Many:One, OneDirection, active)
- power_bi_shap_barplots[split] -> dataset_split_dimension[dataset_split] (Many:One, OneDirection, active)
- power_bi_shap_barplots[model_name] -> base_models_dimension[model_name] (Many:One, OneDirection, active)


In [93]:
def _render_parameters(parameters: list[dict]) -> str:
    header = ("PARAMETERS -- what-if and field parameters, with their filter column, "
               "current value or options, and whether they arrive in filter_context.")
    lines = []
    for p in parameters:
        if p["type"] == "numeric":
            r = p["range"]
            lines.append(
                f"- [numeric, in filter_context] {p['filter_column']} -> value measure: {p['value_measure']}, "
                f"default {p['default']}, range {r['min']}-{r['max']} step {r['step']}"
            )
        elif p["type"] == "field":
            options = ", ".join(f"{o['label']} = {o['field']}" for o in p["options"])
            lines.append(f"- [field, never in filter_context] {p['filter_column']} -> options: {options}")
    return header + "\n\n" + "\n".join(lines)


PARAMETERS = _render_parameters(schema["parameters"])
print(PARAMETERS)

PARAMETERS -- what-if and field parameters, with their filter column, current value or options, and whether they arrive in filter_context.

- [numeric, in filter_context] Conversion Rate[Conversion Rate] -> value measure: Conversion Rate Value, default 0.06, range 0.0-0.1 step 0.005
- [numeric, in filter_context] Churn Rate[Churn Rate] -> value measure: Churn Rate Value, default 0.39, range 0.0-0.6 step 0.01
- [numeric, in filter_context] Conversion Lift per 10% Recall[Conversion Lift per 10% Recall] -> value measure: Conversion Lift per 10% Recall Value, default 0.002, range 0.0-0.01 step 0.001
- [numeric, in filter_context] AOV Lift per 10% Recall[AOV Lift per 10% Recall] -> value measure: AOV Lift per 10% Recall Value, default 0.05, range 0.0-0.26 step 0.01
- [numeric, in filter_context] Churn Reduction per 10% Recall[Churn Reduction per 10% Recall] -> value measure: Churn Reduction per 10% Recall Value, default 0.005, range 0.0-0.02 step 0.001
- [field, never in filter_context] Eva

## BIGQUERY_SCHEMA

Live against the real `agent_safe` dataset (Layer 2 -- real credentials,
ADC resolves to personal identity locally). Checking directly, rather than
assuming: does plain `INFORMATION_SCHEMA.COLUMNS` actually carry column
descriptions, or only structure (name/type/nullability)?

In [94]:
from google.cloud import bigquery

bq = bigquery.Client(project="instacart-ml-model")

cols = list(bq.query("""
    SELECT table_name, column_name, data_type, is_nullable
    FROM `instacart-ml-model.agent_safe.INFORMATION_SCHEMA.COLUMNS`
    ORDER BY table_name, ordinal_position
""").result())

print(f"{len(cols)} columns")
print(cols[0])
print(list(cols[0].keys()))

94 columns
Row(('candidate_reorder_features', 'user_id', 'INT64', 'YES'), {'table_name': 0, 'column_name': 1, 'data_type': 2, 'is_nullable': 3})
['table_name', 'column_name', 'data_type', 'is_nullable']


In [95]:
# Confirmed: no description field in INFORMATION_SCHEMA.COLUMNS. Column
# descriptions are only exposed via the table metadata API (get_table),
# not plain SQL -- check that against a real table with real descriptions.
table_ref = bq.dataset("agent_safe").table("product_order_analysis")
table = bq.get_table(table_ref)

print("table description:", table.description)
print()
for field in table.schema[:5]:
    print(f"{field.name} ({field.field_type}): {field.description}")

table description: Order-line grain (one row per order x product), enriched with precomputed user, user-product, user-aisle, and user-department cumulative behavioral metrics -- reorder ratios, purchase concentrations, day-of-week/time-of-day affinities -- so most product and order questions don't require recomputing them from raw history.

order_id (INTEGER): Unique identifier for the order this line item belongs to.
user_id (INTEGER): Unique identifier for the customer who placed the order.
order_number (INTEGER): Sequential count of this order within the user's order history (1 = their first order).
order_dow (INTEGER): Day of week the order was placed (0 = Sunday through 6 = Saturday).
order_hour_of_day (INTEGER): Hour of day the order was placed, 24-hour format (0-23).


In [96]:
def build_bigquery_schema(client: bigquery.Client, dataset: str) -> str:
    header = "BIGQUERY SCHEMA -- every table in agent_safe, with its columns, types, and descriptions."
    blocks = []
    for tbl_ref in client.list_tables(dataset):
        table = client.get_table(tbl_ref)
        lines = [f"### {table.table_id}"]
        if table.description:
            lines.append(table.description)
        for field in table.schema:
            desc = f": {field.description}" if field.description else ""
            lines.append(f"- {field.name} ({field.field_type}){desc}")
        blocks.append("\n".join(lines))
    return header + "\n\n" + "\n\n".join(blocks)


BIGQUERY_SCHEMA = build_bigquery_schema(bq, "agent_safe")
print(BIGQUERY_SCHEMA)
print("...")
print(f"\n{len(BIGQUERY_SCHEMA)} chars total")

BIGQUERY SCHEMA -- every table in agent_safe, with its columns, types, and descriptions.

### candidate_reorder_features
One row per (user, candidate product, anchor order) -- every product a user has ever ordered, evaluated as a reorder candidate for the order immediately following the anchor, with the reorder outcome label and the engineered features used to predict it. Use for questions correlating features against reorder likelihood; product_order_analysis is the right table for general order/product questions.
- user_id (INTEGER): Unique identifier for the customer this training example belongs to.
- anchor_order_id (INTEGER): Order used as the reference point for this row's cumulative features — the order immediately before the label order.
- anchor_order_number (INTEGER): Sequential order number of the anchor order within this user's order history.
- candidate_product_id (INTEGER): Product being evaluated as a reorder candidate — any product this user has ordered at least once b

## Few-shot examples

Authored content, not rendered from an artifact — each demonstrates a real
composition pattern against the schema it sits next to (`.claude/rules/
gateway.md`'s revised order places DAX examples after `PARAMETERS` and
BigQuery examples after `BIGQUERY_SCHEMA`, not as one trailing block).
Verified live against the real dataset/model before writing anything down;
no output values are kept in the examples themselves, so there's nothing
to mistake for current data.

In [97]:
DAX_FEW_SHOT_EXAMPLES = '''DAX EXAMPLES

Q: "What was the champion model's Recall@5 on the test set?"
EVALUATE
CALCULATETABLE(
    ROW("Champion Recall", [Champion Recall]),
    dataset_split_dimension[dataset_split] = "Test"
)
Note: evaluation_metrics has one row per model per split -- omitting a split filter sums across all of them. Filter dataset_split_dimension[dataset_split] explicitly, from filter_context when present.

Q: "What would the annual profit lift be if churn rate were 45% instead of the current assumption?"
DEFINE VAR ChampionModel = [Champion_Model]
EVALUATE
CALCULATETABLE(
    ROW("Annual Profit Lift", [Annual_Profit_Lift]),
    'Churn Rate'[Churn Rate] = 0.45,
    evaluation_metrics[Model] = ChampionModel,
    dataset_split_dimension[dataset_split] = "Test"
)
Note: a measure can't be a filter's comparison value directly in CALCULATETABLE -- assign it to a VAR first. Scenario questions usually need split, model, and parameter filters together, not just the one being asked about.

Q: "What are the top 5 most important features for the XGBoost Ranker model?"
EVALUATE
TOPN(
    5,
    FILTER(power_bi_shap_barplots, power_bi_shap_barplots[model_name] = "XGBoost Ranker" && power_bi_shap_barplots[split] = "test"),
    power_bi_shap_barplots[mean_absolute_shap], DESC
)
Note: TOPN selects the right rows, but executeQueries doesn't guarantee they arrive sorted -- sort client-side before presenting a ranked list.

Q: "What are the top SHAP features for the champion model?"
Note: power_bi_shap_barplots only covers the four base models -- an ensemble like the champion model has no SHAP decomposition. Use a specific base model, or say none exists, rather than filtering on [Champion_Model] and getting nothing back.'''

print(DAX_FEW_SHOT_EXAMPLES)
print(f"\n{len(DAX_FEW_SHOT_EXAMPLES)} chars total")

DAX EXAMPLES

Q: "What was the champion model's Recall@5 on the test set?"
EVALUATE
CALCULATETABLE(
    ROW("Champion Recall", [Champion Recall]),
    dataset_split_dimension[dataset_split] = "Test"
)
Note: evaluation_metrics has one row per model per split -- omitting a split filter sums across all of them. Filter dataset_split_dimension[dataset_split] explicitly, from filter_context when present.

Q: "What would the annual profit lift be if churn rate were 45% instead of the current assumption?"
DEFINE VAR ChampionModel = [Champion_Model]
EVALUATE
CALCULATETABLE(
    ROW("Annual Profit Lift", [Annual_Profit_Lift]),
    'Churn Rate'[Churn Rate] = 0.45,
    evaluation_metrics[Model] = ChampionModel,
    dataset_split_dimension[dataset_split] = "Test"
)
Note: a measure can't be a filter's comparison value directly in CALCULATETABLE -- assign it to a VAR first. Scenario questions usually need split, model, and parameter filters together, not just the one being asked about.

Q: "What are 

In [98]:
BIGQUERY_FEW_SHOT_EXAMPLES = '''BIGQUERY EXAMPLES

Q: "What's the reorder rate by department?"
SELECT department, ROUND(AVG(reordered), 4) AS reorder_rate, COUNT(*) AS n
FROM `instacart-ml-model.agent_safe.product_order_analysis`
GROUP BY department
ORDER BY reorder_rate DESC
Note: reordered is a 0/1 flag at the order-line grain, so AVG(reordered) grouped by department gives a true per-department reorder rate directly.

Q: "On average, how many distinct products has each user ordered?"
SELECT ROUND(AVG(user_distinct_product_count), 2) AS avg_distinct_products
FROM (
  SELECT DISTINCT user_id, user_distinct_product_count
  FROM `instacart-ml-model.agent_safe.product_order_analysis`
)
Note: user_-prefixed columns (user_distinct_product_count, user_reorder_ratio, user_avg_basket_size, etc.) are precomputed per user and repeat on every order-line row for that user. Averaging one directly over product_order_analysis over-weights users with more order lines -- deduplicate to one row per user first.

Q: "Do users who reorder products more quickly have a higher reorder rate for that product?"
SELECT
  CASE WHEN user_prod_avg_days_between_purchases < 7 THEN 'under 7 days' ELSE '7+ days' END AS pace_bucket,
  ROUND(AVG(label_reordered), 4) AS reorder_rate,
  COUNT(*) AS n
FROM `instacart-ml-model.agent_safe.candidate_reorder_features`
WHERE user_prod_avg_days_between_purchases IS NOT NULL
GROUP BY pace_bucket
Note: candidate_reorder_features is the right table for correlating an engineered feature against reorder likelihood (label_reordered) -- product_order_analysis is order-line grain with no candidate/label structure for this kind of question.'''

print(BIGQUERY_FEW_SHOT_EXAMPLES)
print(f"\n{len(BIGQUERY_FEW_SHOT_EXAMPLES)} chars total")

BIGQUERY EXAMPLES

Q: "What's the reorder rate by department?"
SELECT department, ROUND(AVG(reordered), 4) AS reorder_rate, COUNT(*) AS n
FROM `instacart-ml-model.agent_safe.product_order_analysis`
GROUP BY department
ORDER BY reorder_rate DESC
Note: reordered is a 0/1 flag at the order-line grain, so AVG(reordered) grouped by department gives a true per-department reorder rate directly.

Q: "On average, how many distinct products has each user ordered?"
SELECT ROUND(AVG(user_distinct_product_count), 2) AS avg_distinct_products
FROM (
  SELECT DISTINCT user_id, user_distinct_product_count
  FROM `instacart-ml-model.agent_safe.product_order_analysis`
)
Note: user_-prefixed columns (user_distinct_product_count, user_reorder_ratio, user_avg_basket_size, etc.) are precomputed per user and repeat on every order-line row for that user. Averaging one directly over product_order_analysis over-weights users with more order lines -- deduplicate to one row per user first.

Q: "Do users who reorde

## SYSTEM_INSTRUCTIONS

Authored, not rendered. Goes first in the final bundle order -- role and
behavioral rules established before the model sees any reference material.

In [99]:
SYSTEM_INSTRUCTIONS = """SYSTEM INSTRUCTIONS

You are a senior data professional supporting business stakeholders evaluating a machine learning reorder-recommendation model and its projected financial impact. Translate technical analysis -- SQL, DAX, model evaluation metrics, financial modeling -- into clear, direct language a business audience can act on. Explain a term only when it isn't obvious from context, never for its own sake.

TOOLS
BigQuery answers upstream/warehouse questions -- raw order and product features, pre-model data. DAX answers post-model, dashboard-displayed questions -- model evaluation metrics, financial impact. These domains don't overlap; don't use one where the other is authoritative.

Tools that search or render (search_docs, get_page_info, generate_chart) are never a source of a number themselves.

GROUNDING
Every number in an answer must come from a live run_bigquery_sql or run_dax_query call made this turn. Never state a number from memory, from the background material in this prompt, or from a prior turn's answer -- even one that looks identical to what you'd compute now.

There is no live source for future data, only current and historical figures. Decline requests for forecasts or projections rather than generating one.

Table and measure names come from the registries in this prompt, which are a complete enumeration, not a partial index -- never invent a plausible-looking name. If something needed isn't there, it doesn't exist in this model; say so rather than guessing.

When composing DAX, reference an existing measure by name rather than reconstructing its logic -- only write new calculation logic when no existing measure covers the question.

filter_context and active_page describe what the user is currently looking at and are authoritative -- incorporate them into a DAX query rather than answering against the model's default, unfiltered state. Field parameters (which evaluation metric or ensemble combination is currently displayed) cannot be captured this way and are structurally unknowable -- don't guess or imply you know which one is selected.

CONVERSATION HISTORY
Prior turns' queries are patterns worth adapting for a related follow-up, not results to cite -- their numeric output isn't stored, only the query text and a truncated summary of the answer.

RESPONDING
Write answers in plain Markdown. Suggest 1-3 short, natural follow-up questions when one would genuinely help -- skip it when nothing natural fits. If you can't fully answer within a reasonable number of steps, give the best partial answer available and say plainly that it's partial, rather than presenting it as complete.

An uploaded screenshot, when present, is layout and attention context only -- never read a number off of it."""

print(SYSTEM_INSTRUCTIONS)
print(f"\n{len(SYSTEM_INSTRUCTIONS)} chars total")

SYSTEM INSTRUCTIONS

You are a senior data professional supporting business stakeholders evaluating a machine learning reorder-recommendation model and its projected financial impact. Translate technical analysis -- SQL, DAX, model evaluation metrics, financial modeling -- into clear, direct language a business audience can act on. Explain a term only when it isn't obvious from context, never for its own sake.

TOOLS
BigQuery answers upstream/warehouse questions -- raw order and product features, pre-model data. DAX answers post-model, dashboard-displayed questions -- model evaluation metrics, financial impact. These domains don't overlap; don't use one where the other is authoritative.

Tools that search or render (search_docs, get_page_info, generate_chart) are never a source of a number themselves.

GROUNDING
Every number in an answer must come from a live run_bigquery_sql or run_dax_query call made this turn. Never state a number from memory, from the background material in this pr

## Final assembly

All seven pieces, in the agreed order -- instructions first, then reference
material, DAX examples sitting right after the schema they demonstrate,
BigQuery examples right after theirs.

In [100]:
STATIC_TEXT = "\n\n".join([
    SYSTEM_INSTRUCTIONS,
    ORIENTATION_BUNDLE,
    TABLE_REGISTRY,
    MEASURE_REGISTRY,
    RELATIONSHIPS,
    PARAMETERS,
    DAX_FEW_SHOT_EXAMPLES,
    BIGQUERY_SCHEMA,
    BIGQUERY_FEW_SHOT_EXAMPLES,
])

# Confirm the order landed correctly -- each component's own header should
# appear in this exact sequence.
import re
headers = re.findall(r"^(SYSTEM INSTRUCTIONS|This section is background|TABLE REGISTRY|MEASURE REGISTRY|RELATIONSHIPS|PARAMETERS|DAX EXAMPLES|BIGQUERY SCHEMA|BIGQUERY EXAMPLES)", STATIC_TEXT, re.MULTILINE)
print(" -> ".join(headers))

print(f"\n{len(STATIC_TEXT)} chars, ~{len(STATIC_TEXT) // 4} tokens (rough estimate)")

SYSTEM INSTRUCTIONS -> This section is background -> TABLE REGISTRY -> MEASURE REGISTRY -> RELATIONSHIPS -> PARAMETERS -> DAX EXAMPLES -> BIGQUERY SCHEMA -> BIGQUERY EXAMPLES

47378 chars, ~11844 tokens (rough estimate)


In [ ]:
print(STATIC_TEXT)

SYSTEM INSTRUCTIONS

You are a senior data professional supporting business stakeholders evaluating a machine learning reorder-recommendation model and its projected financial impact. Translate technical analysis -- SQL, DAX, model evaluation metrics, financial modeling -- into clear, direct language a business audience can act on. Explain a term only when it isn't obvious from context, never for its own sake.

TOOLS
BigQuery answers upstream/warehouse questions -- raw order and product features, pre-model data. DAX answers post-model, dashboard-displayed questions -- model evaluation metrics, financial impact. These domains don't overlap; don't use one where the other is authoritative.

Tools that search or render (search_docs, get_page_info, generate_chart) are never a source of a number themselves.

GROUNDING
Every number in an answer must come from a live run_bigquery_sql or run_dax_query call made this turn. Never state a number from memory, from the background material in this pr

: 